# Lab 4 — HIP Programming & Porting CUDA

**ROCm Certification Program — Level 1**

<div style="background:#eef5ff; border-left:5px solid #3b6fd4; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1e3a8a; font-size:1.05em; margin-bottom:8px;">🎯 Objectives</div>
<ul style='margin-bottom:0;'><li>Write a <b>vector-addition</b> HIP kernel from scratch</li><li><b>Port a CUDA application</b> with <code>hipify</code> and apply manual fixes</li><li>Write a <b>wavefront-aware</b> parallel reduction</li></ul>
</div>

## Concept — Warp vs Wavefront

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">The key architectural difference</div>
<p>CUDA uses a <b>warp of 32 threads</b>. AMD GPUs execute threads in <b>wavefronts</b>, whose size depends on the GPU architecture. Many AMD GPUs use 64-thread wavefronts, while others may use 32-thread wavefronts.</p>
</div>

| NVIDIA (CUDA) | AMD (HIP) |
|---------------|-----------|
| Warp = 32 threads | Wavefront size is architecture-dependent |
| `__shfl_down_sync(mask, v, delta)` | `__shfl_down(v, delta)` |
| `__ballot_sync(mask)` | `__ballot(mask)` |
| `warpSize = 32` | use `warpSize` |

<div style="background:#fff7ea; border-left:5px solid #e0a020; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#8a5a06; font-size:1.05em; margin-bottom:8px;">⚠️ Common porting pitfall — don't hardcode execution-group size</div>
<p>Many CUDA applications hardcode a warp size of 32 in reduction, scan, and synchronization code. Such code may compile successfully on AMD GPUs but produce incorrect results or poor performance. Always use <code>warpSize</code> instead of hardcoding the execution-group size.</p>
</div>

## Part 1 — Vector Add from Scratch

<div style="background:#eef9f1; border-left:5px solid #2f9e6e; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1f7a52; font-size:1.05em; margin-bottom:8px;">💡 Why vector addition?</div>
<p>It is the classic first GPU program. Though simple, it shows the full HIP workflow used by every GPU app: allocate device memory → copy data to GPU → launch kernel → copy results back → verify.</p>
</div>

In [ ]:
%%writefile vec_add_hip.cpp
#include <hip/hip_runtime.h>
#include <stdio.h>
#define HIP_CHECK(cmd) do {                              \
    hipError_t e = (cmd);                                \
    if (e != hipSuccess) {                               \
        fprintf(stderr, "[HIP_CHECK] %s:%d  %s\n",        \
            __FILE__, __LINE__, hipGetErrorString(e));    \
        exit(EXIT_FAILURE); } } while (0)
__global__ void vec_add(const float* A, const float* B, float* C, int N) {
    int i = blockDim.x * blockIdx.x + threadIdx.x;
    if (i < N)
        C[i] = A[i] + B[i];
}
int main() {
    const int N  = 1 << 20;            // 1,048,576 elements
    size_t bytes = N * sizeof(float);  // 4 MB per vector
    float *h_A = new float[N], *h_B = new float[N], *h_C = new float[N];
    for (int i = 0; i < N; i++) { h_A[i] = i * 1.0f; h_B[i] = i * 2.0f; }
    float *d_A, *d_B, *d_C;
    HIP_CHECK(hipMalloc(&d_A, bytes));
    HIP_CHECK(hipMalloc(&d_B, bytes));
    HIP_CHECK(hipMalloc(&d_C, bytes));
    HIP_CHECK(hipMemcpy(d_A, h_A, bytes, hipMemcpyHostToDevice));
    HIP_CHECK(hipMemcpy(d_B, h_B, bytes, hipMemcpyHostToDevice));
    vec_add<<<(N + 255) / 256, 256>>>(d_A, d_B, d_C, N);
    HIP_CHECK(hipGetLastError());       // catch bad launch config
    HIP_CHECK(hipDeviceSynchronize());  // catch runtime kernel errors
    HIP_CHECK(hipMemcpy(h_C, d_C, bytes, hipMemcpyDeviceToHost));
    int errs = 0;
    for (int i = 0; i < N; i++)
        if (h_C[i] != i * 3.0f) errs++;
    printf("Vector Add: N=%d, Errors=%d - %s\n", N, errs, errs ? "FAIL":"PASS");
    for (int i = 0; i < 5; i++) printf("  C[%d] = %.1f\n", i, h_C[i]);
    HIP_CHECK(hipFree(d_A)); HIP_CHECK(hipFree(d_B)); HIP_CHECK(hipFree(d_C));
    delete[] h_A; delete[] h_B; delete[] h_C;
    return errs;
}


In [ ]:
!hipcc -o vec_add_hip vec_add_hip.cpp && ./vec_add_hip

## Part 2 — Port CUDA to HIP with hipify

The file below is a CUDA histogram kernel. We port it with `hipify-perl`, then compile and run the result.

In [ ]:
%%writefile histogram_cuda.cu
// histogram_cuda.cu  — ascending bin counts
#include <cuda_runtime.h>
#include <stdio.h>
#include <stdlib.h>

#define NUM_BINS 256

#define CUDA_CHECK(cmd)                                                  \
  do {                                                                   \
    cudaError_t e = (cmd);                                               \
    if (e != cudaSuccess) {                                              \
      fprintf(stderr, "CUDA error %s:%d — %s\n",                        \
              __FILE__, __LINE__, cudaGetErrorString(e));                \
      exit(EXIT_FAILURE);                                                \
    }                                                                    \
  } while (0)

__global__ void histogram(const unsigned char* data, int* bins, int N) {
    __shared__ int local_bins[NUM_BINS];

    for (int i = threadIdx.x; i < NUM_BINS; i += blockDim.x)
        local_bins[i] = 0;
    __syncthreads();

    int idx    = blockIdx.x * blockDim.x + threadIdx.x;
    int stride = blockDim.x * gridDim.x;
    for (int i = idx; i < N; i += stride)
        atomicAdd(&local_bins[data[i]], 1);
    __syncthreads();

    for (int i = threadIdx.x; i < NUM_BINS; i += blockDim.x)
        atomicAdd(&bins[i], local_bins[i]);
}

int main() {
    // N = 0+1+2+...+255 = 255*256/2 = 32640
    const int N = (NUM_BINS - 1) * NUM_BINS / 2;

    // Fill: value v appears exactly v times
    unsigned char* h_data = (unsigned char*)malloc(N * sizeof(unsigned char));
    int pos = 0;
    for (int v = 0; v < NUM_BINS; v++)
        for (int c = 0; c < v; c++)
            h_data[pos++] = (unsigned char)v;

    unsigned char* d_data;
    int*           d_bins;
    CUDA_CHECK(cudaMalloc(&d_data, N * sizeof(unsigned char)));
    CUDA_CHECK(cudaMalloc(&d_bins, NUM_BINS * sizeof(int)));
    CUDA_CHECK(cudaMemset(d_bins, 0, NUM_BINS * sizeof(int)));
    CUDA_CHECK(cudaMemcpy(d_data, h_data, N * sizeof(unsigned char),
                          cudaMemcpyHostToDevice));

    const int threads = 256;
    const int blocks  = (N + threads - 1) / threads;
    histogram<<<blocks, threads>>>(d_data, d_bins, N);
    CUDA_CHECK(cudaDeviceSynchronize());

    int h_bins[NUM_BINS] = {0};
    CUDA_CHECK(cudaMemcpy(h_bins, d_bins, NUM_BINS * sizeof(int),
                          cudaMemcpyDeviceToHost));

    // Verify: bin[v] must equal v
    int errors = 0;
    for (int v = 0; v < NUM_BINS; v++) {
        if (h_bins[v] != v) {
            fprintf(stderr, "MISMATCH bin[%d]: got %d, expected %d\n",
                    v, h_bins[v], v);
            errors++;
        }
    }

    if (errors == 0) {
        printf("bins[0]=%d, bins[1]=%d, bins[2]=%d, ... bins[255]=%d\n",
               h_bins[0], h_bins[1], h_bins[2], h_bins[255]);
    }

    CUDA_CHECK(cudaFree(d_data));
    CUDA_CHECK(cudaFree(d_bins));
    free(h_data);
    return errors > 0 ? EXIT_FAILURE : EXIT_SUCCESS;
}

In [ ]:
# Run hipify to convert CUDA -> HIP
!hipify-perl histogram_cuda.cu -o histogram_hip.cpp 
# -- 2>&1
#hipify-clang histogram_cuda.cu -o histogram_hip.cpp -- 2>&1
print("\n--- Converted file ---")
# !cat histogram_hip.cpp

In [ ]:
!hipcc -o histogram_hip histogram_hip.cpp && ./histogram_hip

<div style="background:#f3f0fb; border-left:5px solid #7c5cd6; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#4c3a8c; font-size:1.05em; margin-bottom:8px;">📝 What did hipify change?</div>
Look at the converted file and list the API mappings.
</div>

| CUDA | HIP (after hipify) |
|------|-------------------|
| `cuda_runtime.h` | `hip/hip_runtime.h` |
| `cudaMalloc` | `hipMalloc` |
| `cudaMemcpy` | `hipMemcpy` |
| `cudaMemset` | `hipMemset` |
| `cudaFree` | `hipFree` |
| `cudaMemcpyHostToDevice` | `hipMemcpyHostToDevice` |

> **Question:** this kernel used `blockDim.x` for strides rather than a hardcoded `warpSize = 32`. Would it have broken if it had used `32` directly? Why?

## Part 3 — Parallel Reduction

<div style="background:#f4f6f9; border-left:5px solid #64748b; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#334155; font-size:1.05em; margin-bottom:8px;">What is a reduction?</div>
<p>A reduction combines many values into one — a sum, max/min, dot product, or statistic. Reductions are among the most common operations in HPC, ML, and data analytics.</p>
</div>

On a GPU, instead of every thread updating one variable, values are combined in a tree so many additions happen in parallel:

```text
Thread 0 + Thread 1
Thread 2 + Thread 3      -->  partial sums  -->  final result
Thread 4 + Thread 5
```

<div style="background:#fff7ea; border-left:5px solid #e0a020; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#8a5a06; font-size:1.05em; margin-bottom:8px;">⚠️ Remember the wavefront size</div>
<p>CUDA reduction examples often assume 32-wide warps. AMD GPUs may use either 32- or 64-thread wavefronts depending on the architecture. Portable HIP code should always use <code>warpSize</code> rather than assuming a particular execution-group size.</p>
</div>

**Exercise goal:** (1) load data into shared memory, (2) perform a block-level reduction, (3) verify the GPU result against a CPU reference.


In [ ]:
%%writefile reduction.cpp

#include <hip/hip_runtime.h>
#include <cstdio>
#include <cstdlib>

#define HIP_CHECK(cmd)                                   \
do {                                                     \
    hipError_t e = cmd;                                  \
    if (e != hipSuccess) {                               \
        printf("HIP error: %s\n",                        \
               hipGetErrorString(e));                    \
        return 1;                                        \
    }                                                    \
} while(0)

//
// Block reduction.
//
// Each block produces one partial sum.
//
__global__ void reduce_sum(
    const float* input,
    float* output,
    int N)
{
    __shared__ float sdata[256];

    int tid = threadIdx.x;
    int gid = blockIdx.x * blockDim.x + tid;

    sdata[tid] =
        (gid < N) ? input[gid] : 0.0f;

    __syncthreads();

    for (int s = blockDim.x / 2; s > 0; s >>= 1)
    {
        if (tid < s)
            sdata[tid] += sdata[tid + s];
        __syncthreads();
    }

    if (tid == 0)
        output[blockIdx.x] = sdata[0];
}

int main()
{
    const int N = 1 << 20;

    const int block = 256;

    float *h_in =
        new float[N];

    float sum_ref = 0.0f;

    for (int i = 0; i < N; i++)
    {
        h_in[i] = 1.0f;
        sum_ref += h_in[i];
    }

    float *d_in;
    float *d_tmp1;
    float *d_tmp2;

    HIP_CHECK(
        hipMalloc(&d_in,
                  N * sizeof(float)));

    HIP_CHECK(
        hipMalloc(&d_tmp1,
                  N * sizeof(float)));

    HIP_CHECK(
        hipMalloc(&d_tmp2,
                  N * sizeof(float)));

    HIP_CHECK(
        hipMemcpy(d_in,
                  h_in,
                  N * sizeof(float),
                  hipMemcpyHostToDevice));

    //
    // Multi-pass reduction.
    //
    int current_size = N;

    float* src = d_in;
    float* dst = d_tmp1;

    while (current_size > 1)
    {
        int grid =
            (current_size + block - 1)
            / block;

        reduce_sum<<<grid, block>>>(
            src,
            dst,
            current_size);

        HIP_CHECK(
            hipGetLastError());

        std::swap(src, dst);

        current_size = grid;
    }

    float result;

    HIP_CHECK(
        hipMemcpy(&result,
                  src,
                  sizeof(float),
                  hipMemcpyDeviceToHost));

    printf("\n=== HIP Reduction ===\n");
    printf("Elements : %d\n", N);
    printf("GPU Sum  : %.0f\n", result);
    printf("CPU Sum  : %.0f\n", sum_ref);
    printf("Status   : %s\n",
           (result == sum_ref)
           ? "PASS"
           : "FAIL");

    HIP_CHECK(hipFree(d_in));
    HIP_CHECK(hipFree(d_tmp1));
    HIP_CHECK(hipFree(d_tmp2));

    delete[] h_in;

    return 0;
}

In [ ]:
!hipcc -o reduction reduction.cpp && ./reduction

---

## Summary

| Task | Status |
|------|--------|
| Vector-add kernel runs and validates | ☐ |
| CUDA histogram ported with hipify and runs | ☐ |
| API mappings identified | ☐ |
| Reduction matches CPU reference | ☐ |

<div style="background:#eef9f1; border-left:5px solid #2f9e6e; padding:16px; border-radius:8px; margin:14px 0; color:#1f2937; line-height:1.6;">
<div style="font-weight:700; color:#1f7a52; font-size:1.05em; margin-bottom:8px;">💡 Key takeaways</div>
<ol style='margin-bottom:0;'><li>The HIP workflow mirrors CUDA — most porting is mechanical via hipify.</li><li>Differences in execution-group size are a common correctness trap when porting CUDA code. Never hardcode the warp or wavefront size.</li><li>Always validate correctness after porting and optimization.</li></ol>
</div>

**Next:** Module 5 — Performance Optimization
